In [1]:
import numpy as np
import pandas as pd

from pathlib import Path
import os
os.chdir("./..")

from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)

# Loading data

In [2]:
alarms = pd.read_csv("data/alarms/alarms_data_preprocessed_v1.csv")
weather = pd.read_csv("data/weather/weather_data_preprocessed_v2.csv")
isw = pd.read_csv("data/isw/isw_data_preprocessed_v3.csv")
telegram = pd.read_csv("data/telegram/telegram_hourly_features_v3.csv")

In [3]:
alarms.head(3)

,id,region_id,region_city,all_region,start,end,duration_min,start_hour,day_of_week,date,month_year
0,52432,12,Львівська обл.,1,2022-02-24 07:43:17,2022-02-24 09:52:28,129.183333,7,thursday,2022-02-24,2022-02
1,53292,23,Чернігівська обл.,1,2022-02-24 14:00:43,2022-02-24 17:11:43,191.000000,14,thursday,2022-02-24,2022-02
2,52080,3,Вінницька обл.,1,2022-02-24 15:40:42,2022-02-24 16:10:42,30.000000,15,thursday,2022-02-24,2022-02


In [4]:
weather.head(3)

,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_preciptype,hour_windspeed,hour_winddir,hour_pressure,hour_visibility,hour_cloudcover,hour_uvindex,hour_conditions,real_hour_datetime,city
0,2.4,-1.6,89.18,0.8,0.0,0.0,['snow'],15.5,275.6,1020.0,0.0,91.5,0.0,Overcast,2022-02-24 00:00:00,Lutsk
1,2.4,-1.5,87.90,0.6,0.0,0.0,['snow'],14.8,280.3,1021.0,0.2,88.2,0.0,Partially cloudy,2022-02-24 01:00:00,Lutsk
2,2.9,-0.8,88.58,1.2,0.0,0.0,['snow'],14.4,310.0,1022.0,10.0,100.0,NaN,Overcast,2022-02-24 02:00:00,Lutsk


In [5]:
isw.head(3)

,date,text_length,cluster,news_count_7d,avg_dist_centroid_7d,topic_entropy_7d,dom_cluster_share_7d,news_velocity_7d,centroid_shift_7d,anomaly_count_7d,news_count_30d,avg_dist_centroid_30d,topic_entropy_30d,dom_cluster_share_30d,news_velocity_30d,centroid_shift_30d,anomaly_count_30d
0,2026-03-10,44967.0,8,7.0,0.155451,0.410116,0.857143,0.0,0.264190,1.0,30.0,0.250153,0.388734,0.9,0.0,0.269645,3.0
1,2026-03-09,34619.0,8,7.0,0.143350,0.410116,0.857143,0.0,0.362958,1.0,30.0,0.243946,0.388734,0.9,0.0,0.289533,3.0
2,2026-03-08,22980.0,8,7.0,0.157616,0.410116,0.857143,0.0,0.399391,1.0,30.0,0.240981,0.388734,0.9,0.0,0.304404,3.0


In [6]:
telegram.head(3)

,datetime,messages_count,has_threat_sum,nlp_артобстрілу,nlp_бпла,nlp_відбій,nlp_відбій_тривоги,nlp_дніпропетровська,nlp_донецька,nlp_запорізька,nlp_нікополь,nlp_нікополь_нікопольська,nlp_нікопольська,nlp_повітряна,nlp_повітряна_тривога,nlp_тривога,nlp_тривоги,nlp_харківська,msg_count_last_3h,msg_count_last_24h,threat_diff_1h,hour_of_day,day_of_week,is_weekend
0,2022-02-24 02:00:00+02:00,1.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1.0,1.0,0.0,2,3,0
1,2022-02-24 03:00:00+02:00,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1.0,1.0,0.0,3,3,0
2,2022-02-24 04:00:00+02:00,1.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2.0,2.0,0.0,4,3,0


# Merge problem

# Alarms data

In [7]:
alarms.head()

,id,region_id,region_city,all_region,start,end,duration_min,start_hour,day_of_week,date,month_year
0,52432,12,Львівська обл.,1,2022-02-24 07:43:17,2022-02-24 09:52:28,129.183333,7,thursday,2022-02-24,2022-02
1,53292,23,Чернігівська обл.,1,2022-02-24 14:00:43,2022-02-24 17:11:43,191.000000,14,thursday,2022-02-24,2022-02
2,52080,3,Вінницька обл.,1,2022-02-24 15:40:42,2022-02-24 16:10:42,30.000000,15,thursday,2022-02-24,2022-02
3,52857,19,Харківська обл.,1,2022-02-24 20:11:47,2022-02-24 20:59:47,48.000000,20,thursday,2022-02-24,2022-02
4,52700,18,Тернопільська обл.,1,2022-02-25 01:59:36,2022-02-25 09:00:19,420.716667,1,friday,2022-02-25,2022-02


In [8]:
alarms.info()

<class 'pandas.DataFrame'>
RangeIndex: 55780 entries, 0 to 55779
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            55780 non-null  int64  
 1   region_id     55780 non-null  int64  
 2   region_city   55780 non-null  str    
 3   all_region    55780 non-null  int64  
 4   start         55780 non-null  str    
 5   end           55780 non-null  str    
 6   duration_min  55780 non-null  float64
 7   start_hour    55780 non-null  int64  
 8   day_of_week   55780 non-null  str    
 9   date          55780 non-null  str    
 10  month_year    55780 non-null  str    
dtypes: float64(1), int64(4), str(6)
memory usage: 9.5 MB


Convert `start`, `end` columns to datetime format

In [9]:
alarms.drop(["start_hour", "day_of_week", "date", "month_year", "duration_min"], axis=1, inplace=True)

In [10]:
alarms["start"] = pd.to_datetime(alarms["start"])
alarms["end"] = pd.to_datetime(alarms["end"])

In [11]:
alarms

,id,region_id,region_city,all_region,start,end
0,52432,12,Львівська обл.,1,2022-02-24 07:43:17,2022-02-24 09:52:28
1,53292,23,Чернігівська обл.,1,2022-02-24 14:00:43,2022-02-24 17:11:43
2,52080,3,Вінницька обл.,1,2022-02-24 15:40:42,2022-02-24 16:10:42
3,52857,19,Харківська обл.,1,2022-02-24 20:11:47,2022-02-24 20:59:47
4,52700,18,Тернопільська обл.,1,2022-02-25 01:59:36,2022-02-25 09:00:19
...,...,...,...,...,...,...
55775,158642,14,Одеська обл.,1,2025-03-01 21:49:30,2025-03-01 23:24:45
55776,158635,9,Київська обл.,1,2025-03-01 22:20:51,2025-03-02 01:38:57
55777,158636,9,Київ,0,2025-03-01 22:52:10,2025-03-02 00:55:18
55778,158617,3,Вінницька обл.,1,2025-03-01 23:26:07,2025-03-02 02:44:07


In [12]:
timeline = pd.date_range(
    alarms.start.min().floor("h"),
    alarms.end.max().ceil("h"),
    freq="h"
)

region_id = alarms.region_id.unique()

In [13]:
spine = pd.MultiIndex.from_product([region_id, timeline], names=["region_id", "time"]) \
            .to_frame(index=False) \
            .sort_values(["region_id", "time"]) \
            .reset_index(drop=True)

In [14]:
spine.head()

,region_id,time
0,1,2022-02-24 07:00:00
1,1,2022-02-24 08:00:00
2,1,2022-02-24 09:00:00
3,1,2022-02-24 10:00:00
4,1,2022-02-24 11:00:00


In [15]:
# Step 1: create a helper column with the list of hours per alarm
alarms["hours"] = alarms.apply(
    lambda row: pd.date_range(row["start"].floor("h"), row["end"].floor("h"), freq="h"),
    axis=1
)

# Step 2: explode — each hour becomes its own row, region_id stays attached
alarm_expanded = alarms[["region_id", "hours"]].explode("hours")
alarm_expanded = alarm_expanded.rename(columns={"hours": "time"})
alarm_expanded["alarm"] = 1

# Step 3: merge onto spine
merged_df = spine.merge(alarm_expanded, on=["region_id", "time"], how="left")
merged_df["alarm"] = merged_df["alarm"].fillna(0).astype(int)

In [16]:
merged_df.loc[merged_df.region_id == 12].head()

,region_id,time,alarm
270540,12,2022-02-24 07:00:00,1
270541,12,2022-02-24 08:00:00,1
270542,12,2022-02-24 09:00:00,1
270543,12,2022-02-24 10:00:00,0
270544,12,2022-02-24 11:00:00,0


In [17]:
weather.head()

,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_preciptype,hour_windspeed,hour_winddir,hour_pressure,hour_visibility,hour_cloudcover,hour_uvindex,hour_conditions,real_hour_datetime,city
0,2.4,-1.6,89.18,0.8,0.0,0.0,['snow'],15.5,275.6,1020.0,0.0,91.5,0.0,Overcast,2022-02-24 00:00:00,Lutsk
1,2.4,-1.5,87.90,0.6,0.0,0.0,['snow'],14.8,280.3,1021.0,0.2,88.2,0.0,Partially cloudy,2022-02-24 01:00:00,Lutsk
2,2.9,-0.8,88.58,1.2,0.0,0.0,['snow'],14.4,310.0,1022.0,10.0,100.0,NaN,Overcast,2022-02-24 02:00:00,Lutsk
3,2.3,-1.3,86.63,0.3,0.0,0.0,['snow'],13.3,295.1,1021.0,0.1,92.0,0.0,Overcast,2022-02-24 03:00:00,Lutsk
4,1.9,-1.8,87.85,0.1,0.0,0.0,['snow'],13.3,305.8,1021.0,0.0,93.8,0.0,Overcast,2022-02-24 04:00:00,Lutsk


In [18]:
regions = pd.read_csv("data/alarms/regions.csv")

regions

,region,center_city_ua,center_city_en,region_alt,region_id
0,АР Крим,Сімферополь,Simferopol,Крим,1
1,Вінницька,Вінниця,Vinnytsia,Вінниччина,2
2,Волинська,Луцьк,Lutsk,Волинь,3
3,Дніпропетровська,Дніпро,Dnipro,Дніпропетровщина,4
4,Донецька,Донецьк,Donetsk,Донеччина,5
5,Житомирська,Житомир,Zhytomyr,Житомирщина,6
6,Закарпатська,Ужгород,Uzhgorod,Закарпаття,7
7,Запорізька,Запоріжжя,Zaporozhye,Запоріжжя,8
8,Івано-Франківська,Івано-Франківськ,Ivano-Frankivsk,Івано-Франківщина,9
9,Київська,Київ,Kyiv,Київщина,10


In [19]:
regions.center_city_en.unique()

<ArrowStringArray>
[     'Simferopol',       'Vinnytsia',           'Lutsk',          'Dnipro',
         'Donetsk',        'Zhytomyr',        'Uzhgorod',      'Zaporozhye',
 'Ivano-Frankivsk',            'Kyiv',   'Kropyvnytskyi',         'Luhansk',
            'Lviv',        'Mykolaiv',           'Odesa',         'Poltava',
           'Rivne',            'Sumy',        'Ternopil',         'Kharkiv',
         'Kherson',    'Khmelnytskyi',        'Cherkasy',      'Chernivtsi',
       'Chernihiv']
Length: 25, dtype: str

In [20]:
weather_df = weather.copy()

In [21]:
weather_df.city.unique()

<ArrowStringArray>
[          'Lutsk',   'Kropyvnytskyi',          'Dnipro',            'Kyiv',
         'Kherson',      'Chernivtsi',       'Chernihiv',           'Odesa',
        'Mykolaiv',         'Kharkiv',    'Khmelnytskyi',         'Donetsk',
        'Uzhgorod',      'Zaporozhye',           'Rivne',        'Zhytomyr',
        'Ternopil',         'Poltava',            'Lviv', 'Ivano-Frankivsk',
        'Cherkasy',            'Sumy',       'Vinnytsia']
Length: 23, dtype: str

In [22]:
regions.loc[~regions.center_city_en.isin(set(weather_df.city.unique()))]

,region,center_city_ua,center_city_en,region_alt,region_id
0,АР Крим,Сімферополь,Simferopol,Крим,1
11,Луганська,Луганськ,Luhansk,Луганщина,12


There is no Crimea and Luhansk in `weather` dataset. But knowing that in this cities permanent alarm status, it is not necessary to know weather there. Model should always predict 1 for alarm status there.

In [23]:
region_id = pd.DataFrame({"city": regions.center_city_en, "region_id": regions.region_id})
weather_df = weather_df.merge(region_id, on=["city"], how="left")

In [24]:
weather_df["time"] = pd.to_datetime(weather_df.pop("real_hour_datetime"))

In [25]:
weather_df

,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_preciptype,hour_windspeed,hour_winddir,hour_pressure,hour_visibility,hour_cloudcover,hour_uvindex,hour_conditions,city,region_id,time
0,2.4,-1.6,89.18,0.8,0.0,0.0,['snow'],15.5,275.6,1020.0,0.0,91.5,0.0,Overcast,Lutsk,3,2022-02-24 00:00:00
1,2.4,-1.5,87.90,0.6,0.0,0.0,['snow'],14.8,280.3,1021.0,0.2,88.2,0.0,Partially cloudy,Lutsk,3,2022-02-24 01:00:00
2,2.9,-0.8,88.58,1.2,0.0,0.0,['snow'],14.4,310.0,1022.0,10.0,100.0,NaN,Overcast,Lutsk,3,2022-02-24 02:00:00
3,2.3,-1.3,86.63,0.3,0.0,0.0,['snow'],13.3,295.1,1021.0,0.1,92.0,0.0,Overcast,Lutsk,3,2022-02-24 03:00:00
4,1.9,-1.8,87.85,0.1,0.0,0.0,['snow'],13.3,305.8,1021.0,0.0,93.8,0.0,Overcast,Lutsk,3,2022-02-24 04:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
608228,-1.3,-3.5,73.09,-5.5,0.0,0.0,NaN,5.8,174.9,1030.0,NaN,100.0,0.0,Overcast,Poltava,16,2025-03-01 19:00:00
608229,-1.8,-1.8,81.17,-4.6,0.0,0.0,NaN,0.0,170.5,1029.6,10.0,100.0,0.0,Overcast,Poltava,16,2025-03-01 20:00:00
608230,-1.0,-3.5,68.31,-6.1,0.0,0.0,NaN,6.8,168.7,1029.0,NaN,99.6,0.0,Overcast,Poltava,16,2025-03-01 21:00:00
608231,-1.7,-4.5,71.36,-6.2,0.0,0.0,NaN,7.2,173.4,1029.0,NaN,98.2,0.0,Overcast,Poltava,16,2025-03-01 22:00:00


In [26]:
merged_df = merged_df.merge(weather_df, on=["region_id", "time"], how="left")

In [27]:
merged_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 643587 entries, 0 to 643586
Data columns (total 18 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   region_id        643587 non-null  int64         
 1   time             643587 non-null  datetime64[us]
 2   alarm            643587 non-null  int64         
 3   hour_temp        590506 non-null  float64       
 4   hour_feelslike   590506 non-null  float64       
 5   hour_humidity    590506 non-null  float64       
 6   hour_dew         590506 non-null  float64       
 7   hour_precip      590408 non-null  float64       
 8   hour_precipprob  590506 non-null  float64       
 9   hour_preciptype  63516 non-null   str           
 10  hour_windspeed   590506 non-null  float64       
 11  hour_winddir     590506 non-null  float64       
 12  hour_pressure    590506 non-null  float64       
 13  hour_visibility  322101 non-null  float64       
 14  hour_cloudcover  590506 non-nul

In [28]:
tg_df = telegram.copy()
tg_df.rename({"datetime": "time"}, axis=1, inplace=True)
tg_df.time = pd.to_datetime(tg_df.time, utc=True) \
                .dt.tz_convert("Europe/Kyiv") \
                .dt.floor("h", ambiguous="infer")

In [29]:
merged_df.time = merged_df.time.dt.tz_localize("Europe/Kyiv", ambiguous="NaT", nonexistent="NaT")

In [30]:
merged_df = merged_df.merge(tg_df, on=["time"], how="left")

In [31]:
merged_df["date"] = merged_df["time"].dt.date

In [32]:
isw.info()

<class 'pandas.DataFrame'>
RangeIndex: 1658 entries, 0 to 1657
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   1658 non-null   str    
 1   text_length            1658 non-null   float64
 2   cluster                1658 non-null   int64  
 3   news_count_7d          1658 non-null   float64
 4   avg_dist_centroid_7d   1658 non-null   float64
 5   topic_entropy_7d       1658 non-null   float64
 6   dom_cluster_share_7d   1658 non-null   float64
 7   news_velocity_7d       1658 non-null   float64
 8   centroid_shift_7d      1658 non-null   float64
 9   anomaly_count_7d       1658 non-null   float64
 10  news_count_30d         1658 non-null   float64
 11  avg_dist_centroid_30d  1658 non-null   float64
 12  topic_entropy_30d      1658 non-null   float64
 13  dom_cluster_share_30d  1658 non-null   float64
 14  news_velocity_30d      1658 non-null   float64
 15  centroid_shift_

In [33]:
isw.date = pd.to_datetime(isw.date)

In [34]:
isw.isna().sum()

date                     0
text_length              0
cluster                  0
news_count_7d            0
avg_dist_centroid_7d     0
topic_entropy_7d         0
dom_cluster_share_7d     0
news_velocity_7d         0
centroid_shift_7d        0
anomaly_count_7d         0
news_count_30d           0
avg_dist_centroid_30d    0
topic_entropy_30d        0
dom_cluster_share_30d    0
news_velocity_30d        0
centroid_shift_30d       0
anomaly_count_30d        0
dtype: int64

In [35]:
isw = isw.loc[~isw.date.isna()]

In [36]:
isw.isna().sum().sum()

np.int64(0)

In [37]:
merged_df.date = pd.to_datetime(merged_df.date)

In [38]:
merged_df = merged_df.merge(isw, on="date", how="left")

In [39]:
merged_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 828788 entries, 0 to 828787
Data columns (total 58 columns):
 #   Column                     Non-Null Count   Dtype                      
---  ------                     --------------   -----                      
 0   region_id                  828788 non-null  int64                      
 1   time                       828644 non-null  datetime64[us, Europe/Kyiv]
 2   alarm                      828788 non-null  int64                      
 3   hour_temp                  760485 non-null  float64                    
 4   hour_feelslike             760485 non-null  float64                    
 5   hour_humidity              760485 non-null  float64                    
 6   hour_dew                   760485 non-null  float64                    
 7   hour_precip                760387 non-null  float64                    
 8   hour_precipprob            760485 non-null  float64                    
 9   hour_preciptype            82937 non-null   str 

In [40]:
merged_df.shape

(828788, 58)

In [41]:
df = merged_df.copy()

In [42]:
df = df.loc[~df.time.isna()]

In [43]:
with pd.option_context("display.max_rows", None):
    print(df.isna().sum())

region_id                         0
time                              0
alarm                             0
hour_temp                     68225
hour_feelslike                68225
hour_humidity                 68225
hour_dew                      68225
hour_precip                   68323
hour_precipprob               68225
hour_preciptype              745709
hour_windspeed                68225
hour_winddir                  68225
hour_pressure                 68225
hour_visibility              421313
hour_cloudcover               68225
hour_uvindex                  74436
hour_conditions               68225
city                          68225
messages_count                    0
has_threat_sum                    0
nlp_артобстрілу                   0
nlp_бпла                          0
nlp_відбій                        0
nlp_відбій_тривоги                0
nlp_дніпропетровська              0
nlp_донецька                      0
nlp_запорізька                    0
nlp_нікополь                

In [44]:
df = df.drop(["date", "city"], axis=1)

In [45]:
df = df.loc[~df.hour_temp.isna()]

df.text_length = df.text_length.fillna(0)
df.hour_precip = df.hour_precip.fillna(0)
df.hour_preciptype = df.hour_preciptype.fillna("None")

df.hour_visibility = df.hour_visibility.ffill()
df.hour_uvindex = df.hour_uvindex.ffill()

In [46]:
with pd.option_context("display.max_rows", None):
    print(df.isna().sum())

region_id                         0
time                              0
alarm                             0
hour_temp                         0
hour_feelslike                    0
hour_humidity                     0
hour_dew                          0
hour_precip                       0
hour_precipprob                   0
hour_preciptype                   0
hour_windspeed                    0
hour_winddir                      0
hour_pressure                     0
hour_visibility                   0
hour_cloudcover                   0
hour_uvindex                      0
hour_conditions                   0
messages_count                    0
has_threat_sum                    0
nlp_артобстрілу                   0
nlp_бпла                          0
nlp_відбій                        0
nlp_відбій_тривоги                0
nlp_дніпропетровська              0
nlp_донецька                      0
nlp_запорізька                    0
nlp_нікополь                      0
nlp_нікополь_нікопольська   

In [47]:
df.loc[(df.cluster.isna() & df.text_length != 0)]

,region_id,time,alarm,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_preciptype,hour_windspeed,hour_winddir,hour_pressure,hour_visibility,hour_cloudcover,hour_uvindex,hour_conditions,messages_count,has_threat_sum,nlp_артобстрілу,nlp_бпла,nlp_відбій,nlp_відбій_тривоги,nlp_дніпропетровська,nlp_донецька,nlp_запорізька,nlp_нікополь,nlp_нікополь_нікопольська,nlp_нікопольська,nlp_повітряна,nlp_повітряна_тривога,nlp_тривога,nlp_тривоги,nlp_харківська,msg_count_last_3h,msg_count_last_24h,threat_diff_1h,hour_of_day,day_of_week,is_weekend,text_length,cluster,news_count_7d,avg_dist_centroid_7d,topic_entropy_7d,dom_cluster_share_7d,news_velocity_7d,centroid_shift_7d,anomaly_count_7d,news_count_30d,avg_dist_centroid_30d,topic_entropy_30d,dom_cluster_share_30d,news_velocity_30d,centroid_shift_30d,anomaly_count_30d


In [48]:
df = df.fillna(-1)

In [49]:
df.isna().sum().sum()

np.int64(0)

In [50]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

cat_cols = list(df.select_dtypes(include=["str"]).columns)
num_cols = list(df.select_dtypes(include=["int", "float"]).columns)

for col_name in cat_cols:
    df[col_name] = encoder.fit_transform(df[[col_name]])

C:\PythonProjects\DS_lab\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:120: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\PythonProjects\DS_lab\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:120: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [51]:
df.head(3)

,region_id,time,alarm,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_preciptype,hour_windspeed,hour_winddir,hour_pressure,hour_visibility,hour_cloudcover,hour_uvindex,hour_conditions,messages_count,has_threat_sum,nlp_артобстрілу,nlp_бпла,nlp_відбій,nlp_відбій_тривоги,nlp_дніпропетровська,nlp_донецька,nlp_запорізька,nlp_нікополь,nlp_нікополь_нікопольська,nlp_нікопольська,nlp_повітряна,nlp_повітряна_тривога,nlp_тривога,nlp_тривоги,nlp_харківська,msg_count_last_3h,msg_count_last_24h,threat_diff_1h,hour_of_day,day_of_week,is_weekend,text_length,cluster,news_count_7d,avg_dist_centroid_7d,topic_entropy_7d,dom_cluster_share_7d,news_velocity_7d,centroid_shift_7d,anomaly_count_7d,news_count_30d,avg_dist_centroid_30d,topic_entropy_30d,dom_cluster_share_30d,news_velocity_30d,centroid_shift_30d,anomaly_count_30d
34071,2,2022-02-24 07:00:00+02:00,0,1.5,1.5,93.73,0.6,0.0,0.0,0,4.3,291.8,1023.0,24.1,96.4,0.0,3,21.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,45.0,47.0,2.0,7.0,3.0,0.0,0.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
34072,2,2022-02-24 08:00:00+02:00,0,1.4,-0.9,95.09,0.7,0.3,100.0,3,7.2,310.0,1024.0,13.0,90.0,1.0,15,20.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,48.0,67.0,0.0,8.0,3.0,0.0,0.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
34073,2,2022-02-24 09:00:00+02:00,0,1.9,-0.6,90.43,0.5,0.0,0.0,0,8.3,299.3,1025.0,24.1,92.7,1.0,3,19.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,60.0,86.0,-1.0,9.0,3.0,0.0,0.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0


In [52]:
df.shape

(760419, 56)

In [53]:
scaler = StandardScaler()

num_cols = list(df.select_dtypes(include=["int", "float"]).columns)

df_scaled = pd.DataFrame(scaler.fit_transform(df[num_cols]), columns=num_cols)

In [56]:
df_scaled["time"] = df["time"]

In [57]:
df_scaled.shape

(760419, 56)

In [59]:
save_path = Path("data/merged_v1.csv")
df_scaled.to_csv(save_path, index=False)